# Store Sales — fresh 3-day mean for the near horizons, on the 0.39079 blend

**This differs from the 0.39079 submission in exactly one way: the three near-horizon direct
models (`lag ≥ 1/2/4`, serving forecast days 1–4) gain a 61st feature — `rmean_3`, the mean of
the freshest three days each bucket may legally see.** The far bucket (`lag ≥ 16`, serving days
5–16), the recursive half, the 70/30 blend, the five seeds and the dormancy rule are untouched.

## Why this feature, and why only in the near buckets

Every rolling window in the shipped feature set is a multiple of 7 — `rmean_7/14/28/56/112` —
and that is deliberate: a 7-day window holds exactly one of each weekday, so its value does not
depend on where in the week it stands. A 3-day window spans three *consecutive* days and is
weekday-contaminated (Sunday outsells Thursday by 63%).

Per-horizon models changed the calculus. At `lag ≥ 16` a 3-day mean covers days −16..−18 —
stale, redundant with the 7-day version. At `lag ≥ 1` it covers days **−1..−3**: genuinely
fresh information no other feature carries. The model agrees on both counts — where the window
is fresh it promotes `rmean_3` into the top 10 by gain (#4–#9); where stale it ignores it
(#19–#24, 0.1%). Adding it to the far bucket was measured and cancels the gain; hence
near-buckets-only.

## The evidence, and its calibration

Scored on the rows the change actually serves (days 1–4, 7,128 rows), paired across the five
production seeds:

| seed | without | with `rmean_3` | Δ |
|---|---|---|---|
| 42 | 0.37895 | 0.37558 | −0.00337 |
| 79 | 0.37530 | 0.37289 | −0.00241 |
| 116 | 0.37623 | 0.37533 | −0.00089 |
| 153 | 0.37716 | 0.37388 | −0.00327 |
| 190 | 0.37573 | 0.37347 | −0.00226 |

**Mean −0.00244, sd 0.00100, 5/5 seeds, 5.5 σ** — the same evidence band as the changes that
shipped and transferred (per-horizon 6.6 σ, dormant-zero 4.5 σ), not the 1.3 σ band of the
mixed-objective test that just failed. And the check that failure taught: **the 5-seed ensemble
keeps the gain** (−0.00236) instead of absorbing it.

Expected whole-submission effect is small — the near days are 4 of 16, and the direct half
carries 70% of the blend — roughly **−0.0005 locally**. But the change is *new information*
(a window the model previously could not see at all), the one category that has transferred at
or above its measured value here (per-horizon 4.4×, promo rework sign-reversed in our favour),
never below it.

---

Two forecasters that fail in opposite directions, averaged in log space.

**The direct half** is this project's production model: four LightGBM models splitting the
16-day horizon, each using the freshest sales history its own days can legally rely on
(`lag ≥ 1 / 2 / 4 / 16`), 5 seeds each, plus a hard zero for product lines with no sales in
365 days. Leaderboard 0.39586.

**The recursive half** is the architecture behind the strongest public notebooks: one model per
product family, trained to predict a single day ahead from lags 1–63, then rolled forward
sixteen times feeding each prediction back as the next day's lag.

## Why blend these two specifically

Three earlier diversity attempts failed, each for a diagnosable reason. A blend partner has to
be **both decorrelated and comparable in quality**, and until now nothing was:

| Candidate | Residual corr. with ours | Solo score | Outcome |
|---|---|---|---|
| XGBoost | 0.974–0.987 | 0.393–0.406 | correlated — nothing to diversify |
| Lag-depth ensemble | 0.972–0.981 | ~0.398 | same axis as seeds |
| Training-window ensemble | 0.981–0.991 | 0.393–0.400 | same axis as seeds |
| TiDE (deep learning) | 0.790 | 0.499 | decorrelated but 0.11 behind |
| Linear on lags/Fourier | — | 0.462 | 0.07 behind |
| **Recursive per-family** | **0.916** | **0.399** | ✅ **both** |

For reference, two *identical* models differing only by random seed correlate at 0.972–0.979.
The recursive model sits **below** that — it is more different from our model than our own
model is from itself.

## The complementarity is mechanical

| Forecast day | Direct model | Recursive model | |
|---|---|---|---|
| 1 | **0.37977** | 0.38493 | direct wins — observed `lag_1`, tuned, 5 seeds |
| 2 | **0.36721** | 0.38054 | direct wins |
| 4 | **0.37196** | 0.38283 | direct wins |
| 8 | 0.39588 | **0.38634** | recursive wins |
| 12 | 0.40326 | 0.43372 | direct wins |
| **16** | 0.45259 | **0.41185** | **recursive wins by 0.041** |

The direct model is constrained by legality: a day 16 steps out may only use sales at least 16
days old — observed, but stale. The recursive model has `lag_1` at every horizon, of *predicted*
values, so it carries the recent trajectory forward and degrades gracefully exactly where the
direct model falls off a cliff.

**This also serves as the leakage audit.** A recursive model that were somehow seeing the future
would not degrade with horizon at all. This one goes from 0.385 on near days to 0.41–0.43 on far
ones — the signature of compounding error, which is what it should be.

## What was measured

Paired across our 3 seeds (the recursive half is deterministic — stock `LGBMRegressor` does no
subsampling, so `random_state` has no effect and all variation comes from the direct half):

| | |
|---|---|
| direct model alone | 0.39028 |
| recursive alone | 0.39886 |
| **blend, w(direct) = 0.70** | **0.38569** |
| **mean gain** | **−0.00682** (sd 0.00165, 3/3 seeds, ~4.1 σ) |

Larger than either change shipped earlier this week (dormant zero −0.00274, per-horizon
−0.00238).

**Horizon-dependent weights were tested and add nothing:** flat weight −0.00682 (1 parameter),
per-bucket −0.00688 (4), linear ramp −0.00690 (2). Differences of 0.00006–0.00008 are noise.
A full per-horizon weighting looks better (−0.00830) but that is 16 free parameters fitted on
the same rows being scored — overfitting, not signal.

**Why w = 0.70 rather than the sweep's optimum of 0.60.** The optimum scores 0.38553 against
0.70's 0.38569 — a 0.00016 difference, inside noise. `w` was itself chosen on the holdout being
reported, so the honest move is to hedge toward the half with three independent leaderboard
confirmations behind it rather than squeeze the last noise-sized fraction.

**Runtime ≈ 3 hours.** No GPU needed.

In [1]:
import gc, time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

pd.set_option("display.width", 140)

COMP = "store-sales-time-series-forecasting"
REQUIRED = {"train.csv", "test.csv", "stores.csv", "holidays_events.csv"}

def has_data(p: Path) -> bool:
    try:
        return p.is_dir() and REQUIRED.issubset({f.name for f in p.iterdir() if f.is_file()})
    except OSError:
        return False

def find_data() -> Path:
    root = Path("/kaggle/input")
    if root.is_dir():
        for cand in [root / COMP, *sorted(d for d in root.iterdir() if d.is_dir())]:
            if has_data(cand):
                return cand
    for cand in (Path("data"), Path("../data"), Path("../../data")):
        if has_data(cand):
            return cand
    try:
        import kagglehub
        got = Path(kagglehub.competition_download(COMP))
        if has_data(got):
            return got
        for sub in got.rglob("*"):
            if has_data(sub):
                return sub
    except Exception as exc:
        print(f"kagglehub fallback failed: {exc}")
    raise FileNotFoundError(
        f"Could not find {sorted(REQUIRED)}. In the Kaggle editor: + Add Input -> "
        "Competitions -> store-sales-time-series-forecasting.")

DATA = find_data()
ON_KAGGLE = Path("/kaggle/working").is_dir()
OUT = Path("/kaggle/working") if ON_KAGGLE else Path("submissions/blend")
OUT.mkdir(parents=True, exist_ok=True)

HORIZON = 16
MODEL_START = pd.Timestamp("2015-01-01")
EQ_START, EQ_END = pd.Timestamp("2016-04-16"), pd.Timestamp("2016-04-22")
DORMANT_DAYS = 365
N_LAGS_REC = 63          # recursive half: target lags 1..63
BLEND_W = 0.70           # direct model's share, in log space -- see the note above

# The last validated submission's chain-wide total, as a final sanity check. A blend bug once
# shipped half this volume and scored 0.66126 because nobody read the number before uploading.
VALIDATED_VOLUME = 12_635_225      # the 0.39079 blend submission's chain-wide total

DTYPES = {"store_nbr": "int8", "family": "category", "onpromotion": "int32", "sales": "float32"}
train = pd.read_csv(DATA / "train.csv", parse_dates=["date"], dtype=DTYPES)
test = pd.read_csv(DATA / "test.csv", parse_dates=["date"], dtype=DTYPES)
stores = pd.read_csv(DATA / "stores.csv", dtype={"store_nbr": "int8"})
hol = pd.read_csv(DATA / "holidays_events.csv", parse_dates=["date"])

TRAIN_END = train.date.max()
VALID_START = TRAIN_END - pd.Timedelta(days=HORIZON - 1)
print(f"train {train.date.min():%Y-%m-%d} -> {TRAIN_END:%Y-%m-%d} | "
      f"test {test.date.min():%Y-%m-%d} -> {test.date.max():%Y-%m-%d}")

train 2013-01-01 -> 2017-08-15 | test 2017-08-16 -> 2017-08-31


---
## Shared panel: Christmas restored, earthquake repaired

Both halves train on the same cleaned history, so the blend is comparing forecasting strategies rather than data preparation.

In [2]:
t0 = time.time()
full_idx = pd.date_range(train.date.min(), test.date.max(), freq="D")

both = pd.concat([train.drop(columns="sales"), test], ignore_index=True)
both["family"] = both.family.astype(str)
sales_w = (train.assign(family=train.family.astype(str))
           .pivot(index="date", columns=["store_nbr", "family"], values="sales")
           .reindex(full_idx).sort_index(axis=1))
promo_w = (both.pivot(index="date", columns=["store_nbr", "family"], values="onpromotion")
           .reindex(full_idx).sort_index(axis=1).fillna(0.0))
xmas = pd.DatetimeIndex([d for d in full_idx
                         if d <= TRAIN_END and d not in set(train.date.unique())])
sales_w.loc[xmas] = 0.0
assert sales_w.loc[:TRAIN_END].isna().sum().sum() == 0

def repair_window(W, start, end, halo_weeks=8):
    out = W.copy(); win = pd.date_range(start, end)
    ctx = W.loc[start - pd.Timedelta(weeks=halo_weeks): end + pd.Timedelta(weeks=halo_weeks)]
    ctx = ctx.drop(index=win, errors="ignore")
    by_dow = ctx.groupby(ctx.index.dayofweek).median()
    for d in win:
        out.loc[d] = by_dow.loc[d.dayofweek].values
    return out

sales_w = repair_window(sales_w, EQ_START, EQ_END)
eq_dates = pd.date_range(EQ_START, EQ_END)
SERIES = sales_w.columns
n_s = len(SERIES)
L = np.log1p(sales_w).astype("float32")
P = np.log1p(promo_w).astype("float32")
ZERO = (sales_w == 0).astype("float32")
print(f"panel {sales_w.shape}, {n_s} series  ({time.time()-t0:.0f}s)")

panel (1704, 1782), 1782 series  (43s)


In [3]:
h = hol.copy()
h = h[~((h.type == "Holiday") & (h.transferred))]
h.loc[h.type == "Transfer", "type"] = "Holiday"
work_days = set(h.loc[h.type == "Work Day", "date"])
h = h[h.type != "Work Day"]
events = h[h.type == "Event"]; h = h[h.type != "Event"]
nat = pd.DatetimeIndex(sorted(set(h.loc[h.locale == "National", "date"])))
nat_name = (h[h.locale == "National"].drop_duplicates("date")[["date", "description"]]
            .rename(columns={"description": "nat_hol_name"}))
loc_name = (h[h.locale == "Local"].drop_duplicates(["date", "locale_name"])
            [["date", "locale_name", "description"]]
            .rename(columns={"locale_name": "city", "description": "loc_hol_name"}))
geo = stores.set_index("store_nbr")[["city", "state", "type", "cluster"]]
geo.columns = ["city", "state", "store_type", "cluster"]
assert not (set(loc_name.city) - set(stores.city))

pos = np.searchsorted(nat.values, full_idx.values)
prev_d = np.where(pos > 0, (full_idx.values - nat.values[np.maximum(pos-1, 0)])
                  / np.timedelta64(1, "D"), 999)
next_d = np.where(pos < len(nat), (nat.values[np.minimum(pos, len(nat)-1)] - full_idx.values)
                  / np.timedelta64(1, "D"), 999)
cal_hol = pd.DataFrame({"date": full_idx,
                        "work_day": full_idx.isin(work_days).astype("int8"),
                        "is_event": full_idx.isin(set(events.date)).astype("int8"),
                        "days_since_nat": np.clip(prev_d, 0, 30).astype("int16"),
                        "days_to_nat": np.clip(next_d, 0, 30).astype("int16")})
print(f"{len(nat_name)} national + {len(loc_name)} local holiday dates")

102 national + 147 local holiday dates


---
# Half 1 — the direct per-horizon model

Unchanged from the 0.39586 submission. Four models, each using the freshest lag its horizon
range legally allows.

**The legality rule, asserted rather than trusted.** A target date `d` at horizon `h` has
forecast origin `o = d − h`; feature `lag_k` is `sales(d − k)`, known at the origin iff
`d − k ≤ d − h`, i.e. **`k ≥ h`**.

In [4]:
promo_feats = {}
promo_feats["promo"] = P
promo_feats["promo_rmean_7"] = P.rolling(7, min_periods=1).mean()
promo_feats["promo_rmean_28"] = P.rolling(28, min_periods=3).mean()
promo_feats["promo_lag_16"] = P.shift(HORIZON)
promo_feats["promo_rel_112"] = P - P.rolling(112, min_periods=14).mean()
promo_feats["promo_rel_28"] = P - P.rolling(28, min_periods=5).mean()
for k in (1, 2, 3, 7):
    promo_feats[f"promo_lead_{k}"] = P.shift(-k)
promo_feats["promo_fwd7"] = P.shift(-6).rolling(7, min_periods=1).mean()
chain = P.mean(axis=1); ones = np.ones(n_s, dtype="float32")
promo_feats["promo_chain_level"] = pd.DataFrame(
    np.outer(chain.to_numpy(dtype="float32"), ones), index=full_idx, columns=SERIES)
promo_feats["promo_chain_rel"] = pd.DataFrame(
    np.outer((chain - chain.rolling(112, min_periods=14).mean()).to_numpy(dtype="float32"),
             ones), index=full_idx, columns=SERIES)
del chain; gc.collect()
praw = np.expm1(P)
fam_p = np.log1p(praw.T.groupby(level=1).sum().T)
sto_p = np.log1p(praw.T.groupby(level=0).sum().T)
def _b(a, lv): return a[SERIES.get_level_values(lv)].set_axis(SERIES, axis=1)
promo_feats["fam_promo_rel"] = _b(fam_p - fam_p.rolling(112, min_periods=14).mean(), 1)
promo_feats["fam_promo_fwd7"] = _b(fam_p.shift(-6).rolling(7, min_periods=1).mean()
                                   - fam_p.rolling(112, min_periods=14).mean(), 1)
promo_feats["store_promo_rel"] = _b(sto_p - sto_p.rolling(112, min_periods=14).mean(), 0)
del praw, fam_p, sto_p; gc.collect()

A_pos = (sales_w.to_numpy() > 0)
gap = np.empty(A_pos.shape, dtype="float32"); _last = np.full(A_pos.shape[1], -999.0)
for i in range(A_pos.shape[0]):
    gap[i] = i - _last
    _last = np.where(A_pos[i], float(i), _last)
GAP = pd.DataFrame(np.minimum(gap, 999.0), index=sales_w.index, columns=SERIES)
del gap, A_pos; gc.collect()

mask = full_idx >= MODEL_START
dates_sel = full_idx[mask]
LAG_OFFSETS = (0, 1, 2, 3, 4, 5, 6, 12, 19, 33, 47)

def build_design(min_lag):
    feats = dict(promo_feats)
    for d in LAG_OFFSETS:
        feats[f"lag_{min_lag + d}"] = L.shift(min_lag + d)
    base = L.shift(min_lag)
    # rmean_3 only where the window is fresh (days -1..-3 at lag>=1): in the far bucket it is
    # stale, weekday-contaminated and measured to cancel the near-bucket gain.
    windows = (3, 7, 14, 28, 56, 112) if min_lag < 16 else (7, 14, 28, 56, 112)
    for w in windows:
        feats[f"rmean_{w}"] = base.rolling(w, min_periods=max(2, w // 4)).mean()
    for w in (14, 28):
        feats[f"rstd_{w}"] = base.rolling(w, min_periods=max(2, w // 4)).std()
    feats["rmax_28"] = base.rolling(28, min_periods=7).max()
    dow_start = 7 * int(np.ceil(min_lag / 7))
    feats["dow_mean_4"] = sum(L.shift(dow_start + 7 * k) for k in range(4)) / 4
    feats["dow_mean_8"] = sum(L.shift(dow_start + 7 * k) for k in range(8)) / 8
    feats["zfrac_28"] = ZERO.shift(min_lag).rolling(28, min_periods=7).mean()
    feats["zfrac_112"] = ZERO.shift(min_lag).rolling(112, min_periods=28).mean()
    feats["days_since_sale"] = GAP.shift(min_lag)
    d = pd.DataFrame({
        "date": np.repeat(dates_sel.values, n_s),
        "store_nbr": np.tile(SERIES.get_level_values(0).to_numpy(), len(dates_sel)),
        "family": np.tile(SERIES.get_level_values(1).to_numpy(), len(dates_sel))})
    for nm, W in feats.items():
        d[nm] = W.to_numpy(dtype="float32")[mask].ravel()
    d["target"] = L.to_numpy(dtype="float32")[mask].ravel()
    del feats; gc.collect()
    d = d.join(geo, on="store_nbr")
    dt = d.date
    d["dow"] = dt.dt.dayofweek.astype("int8"); d["day"] = dt.dt.day.astype("int8")
    d["month"] = dt.dt.month.astype("int8"); d["year"] = dt.dt.year.astype("int16")
    d["dayofyear"] = dt.dt.dayofyear.astype("int16")
    d["is_weekend"] = (d.dow >= 5).astype("int8")
    d["days_to_month_end"] = (dt.dt.days_in_month - dt.dt.day).astype("int8")
    d["payday_window"] = (dt.dt.day.isin([15,16,17,1,2,3]) | (d.days_to_month_end <= 1)).astype("int8")
    d = d.merge(cal_hol, on="date", how="left").merge(nat_name, on="date", how="left")
    d = d.merge(loc_name, on=["date", "city"], how="left")
    d["nat_hol_name"] = d.nat_hol_name.fillna("none")
    d["loc_hol_name"] = d.loc_hol_name.fillna("none")
    for c in ("family","city","state","store_type","nat_hol_name","loc_hol_name"):
        d[c] = d[c].astype("category")
    d["store_nbr"] = d.store_nbr.astype("int16"); d["cluster"] = d.cluster.astype("int16")
    cols = [c for c in d.columns if c not in ("date", "target")]
    want = 61 if min_lag < 16 else 60
    assert len(cols) == want, f"lag>={min_lag}: expected {want} features, got {len(cols)}"
    return d, cols

CATS = ["family","city","state","store_type","nat_hol_name","loc_hol_name"]
PARAMS = dict(objective="regression", metric="rmse", learning_rate=0.08, num_leaves=96,
              min_data_in_leaf=50, feature_fraction=0.75, bagging_fraction=0.8,
              bagging_freq=1, lambda_l2=1.0, feature_pre_filter=False,
              num_threads=0, verbose=-1, seed=42)
SEED_CFG = [(42,0.75,0.80), (79,0.60,0.90), (116,0.85,0.70), (153,0.70,0.85), (190,0.65,0.75)]
def seed_params(i):
    p = dict(PARAMS)
    p["seed"], p["feature_fraction"], p["bagging_fraction"] = SEED_CFG[i]
    return p
def rmsle(yt, yp):
    yp = np.clip(np.asarray(yp, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(yp) - np.log1p(np.asarray(yt, float))) ** 2)))

ASSEMBLY = {1:1, 2:2, 3:4, 4:4, **{h: 16 for h in range(5, 17)}}
MIN_LAGS = sorted(set(ASSEMBLY.values()))
for day, ml in ASSEMBLY.items():
    assert ml >= day, f"day {day} cannot use sales only {ml} days old"
print(f"{len(MIN_LAGS)} direct models {MIN_LAGS} covering 16 forecast days")

4 direct models [1, 2, 4, 16] covering 16 forecast days


In [5]:
val_log_direct, best_rounds = {}, {}
for min_lag in MIN_LAGS:
    t = time.time()
    d, cols = build_design(min_lag)
    eq_row = d.date.isin(eq_dates)
    tr_m = (d.date < VALID_START) & d.target.notna() & ~eq_row
    va_m = (d.date >= VALID_START) & (d.date <= TRAIN_END)
    day_of = ((d.loc[va_m,"date"].to_numpy() - np.datetime64(VALID_START))
              / np.timedelta64(1,"D")).astype(int) + 1
    serves = np.isin(day_of, [dd for dd, ml in ASSEMBLY.items() if ml == min_lag])
    es = np.zeros(len(d), dtype=bool)
    es[np.flatnonzero(va_m.to_numpy())[np.flatnonzero(serves)]] = True
    dtr = lgb.Dataset(d.loc[tr_m, cols], d.loc[tr_m,"target"],
                      categorical_feature=CATS, free_raw_data=False)
    dva = lgb.Dataset(d.loc[es, cols], d.loc[es,"target"],
                      categorical_feature=CATS, reference=dtr, free_raw_data=False)
    m0 = lgb.train(seed_params(0), dtr, num_boost_round=2400, valid_sets=[dva],
                   callbacks=[lgb.early_stopping(150, verbose=False)])
    best_rounds[min_lag] = m0.num_trees()
    logs = [m0.predict(d.loc[va_m, cols])]
    for i in range(1, len(SEED_CFG)):
        mi = lgb.train(seed_params(i), dtr, num_boost_round=best_rounds[min_lag])
        logs.append(mi.predict(d.loc[va_m, cols]))
        del mi; gc.collect()
    val_log_direct[min_lag] = np.mean(logs, axis=0)
    if min_lag == 16:
        y_va = np.expm1(d.loc[va_m,"target"].to_numpy())
        DAY_OF = day_of
    print(f"  direct lag>={min_lag:>2}: {best_rounds[min_lag]:>5} trees ({time.time()-t:.0f}s)",
          flush=True)
    del d, dtr, dva, m0, logs; gc.collect()

direct_val = np.empty(len(DAY_OF))
for day, ml in ASSEMBLY.items():
    sel = DAY_OF == day
    direct_val[sel] = val_log_direct[ml][sel]
print(f"\ndirect half, holdout: {rmsle(y_va, np.expm1(direct_val)):.5f}")

  direct lag>= 1:   826 trees (668s)
  direct lag>= 2:   565 trees (499s)
  direct lag>= 4:  1135 trees (837s)
  direct lag>=16:  1388 trees (1018s)

direct half, holdout: 0.38899


---
# Half 2 — the recursive per-family model

One model per product family (33 of them), each trained across that family's 54 store-series to
predict **one day ahead** from target lags 1–63, then rolled forward sixteen times with each
prediction written back as the next day's `lag_1`.

Three choices reproduced from the public architecture because they are load-bearing there:

- **Per-family partitioning.** This project's own backtests found per-family *helps* recursive
  models and *hurts* direct ones — opposite signs, so it cannot be ported blindly in either
  direction.
- **`log1p` then per-series MinMax.** Fifty-four series share one small model, so putting them
  on a common 0–1 scale lets it fit shape rather than level. (Tested on the *direct* model,
  where it lost by 6.5 σ — the same ingredient is right here and wrong there.)
- **Stock LightGBM, 100 trees, no early stopping.** Note this makes the recursive half
  **deterministic**: the defaults do no row or column subsampling, so `random_state` has no
  effect. There is no seed ensemble on this side, and unexploited diversity remains.

Deliberately *not* copied: `oil` and `transactions` (rejected here five ways and one way
respectively, and the source notebook runs no ablation, so they are suspects rather than proven
ingredients) and the pruned 7-holiday encoding (we measured that holiday *names* win).

In [6]:
cal = pd.DataFrame(index=full_idx)
cal["dow"] = full_idx.dayofweek; cal["day"] = full_idx.day
cal["month"] = full_idx.month; cal["dayofyear"] = full_idx.dayofyear
cal["is_weekend"] = (full_idx.dayofweek >= 5).astype(int)
cal["days_to_month_end"] = full_idx.days_in_month - full_idx.day
cal["payday_window"] = (full_idx.day.isin([15,16,17,1,2,3]) | (cal.days_to_month_end <= 1)).astype(int)
cal["work_day"] = full_idx.isin(work_days).astype(int)
cal["is_event"] = full_idx.isin(set(events.date)).astype(int)
cal["days_since_nat"] = np.clip(prev_d, 0, 30)
cal["days_to_nat"] = np.clip(next_d, 0, 30)
cal["nat_hol"] = pd.Categorical(
    nat_name.set_index("date")["nat_hol_name"].reindex(full_idx)).codes
cal_np = cal.to_numpy(dtype="float32")
date_pos = {d: i for i, d in enumerate(full_idx)}

geo_num = geo.copy()
for c in ("city", "state", "store_type"):
    geo_num[c] = geo_num[c].astype("category").cat.codes
FAMILIES = sorted(SERIES.get_level_values(1).unique())
eq_set = set(pd.date_range(EQ_START, EQ_END))

def run_family(fam, fit_end, target_dates):
    # Train on this family's 54 store-series up to fit_end, then roll forward over target_dates.
    cols = [c for c in SERIES if c[1] == fam]
    Lf = L[cols].to_numpy(dtype="float32")
    Pf = P[cols].to_numpy(dtype="float32")
    n_d, n_st = Lf.shape
    st_arr = np.array([c[0] for c in cols])
    stat = geo_num.loc[st_arr, ["city","state","store_type","cluster"]].to_numpy(dtype="float32")
    stat = np.column_stack([st_arr.astype("float32"), stat])

    # Per-series MinMax fitted on training data only -- never on the window being forecast.
    fit_hi = date_pos[fit_end] + 1
    lo = Lf[:fit_hi].min(axis=0); hi = Lf[:fit_hi].max(axis=0)
    rng = np.where(hi - lo > 1e-9, hi - lo, 1.0)
    S = (Lf - lo) / rng

    start = max(date_pos[MODEL_START], N_LAGS_REC)
    rows_d = np.array([di for di in range(start, fit_hi) if full_idx[di] not in eq_set])

    def make_X(dis, panel):
        n = len(dis)
        lagm = np.empty((n, n_st, N_LAGS_REC), dtype="float32")
        for k in range(1, N_LAGS_REC + 1):
            lagm[:, :, k-1] = panel[dis - k]
        lagm = lagm.reshape(n * n_st, N_LAGS_REC)
        promo = Pf[dis].reshape(-1, 1)
        leads = np.column_stack([Pf[np.minimum(dis + k, n_d - 1)].ravel() for k in (1,2,3,7)])
        return np.column_stack([lagm, promo, leads,
                                np.repeat(cal_np[dis], n_st, axis=0),
                                np.tile(stat, (n, 1))])

    X = make_X(rows_d, S); yv = S[rows_d].ravel()
    ok = np.isfinite(X).all(axis=1) & np.isfinite(yv)
    m = lgb.LGBMRegressor(n_estimators=100, random_state=0, verbose=-1, n_jobs=-1)
    m.fit(X[ok], yv[ok])

    Sw = S.copy()
    out = np.empty((len(target_dates), n_st), dtype="float32")
    for step, dd in enumerate(target_dates):
        di = date_pos[dd]
        p = m.predict(make_X(np.array([di]), Sw))
        Sw[di] = p                      # feed the prediction back as tomorrow's lag_1
        out[step] = p
    return out * rng + lo, cols         # invert the scaler, still in log space

In [7]:
va_dates = pd.date_range(VALID_START, TRAIN_END)
t = time.time()
rec_val_frame = pd.DataFrame(index=va_dates, columns=SERIES, dtype="float32")
for i, fam in enumerate(FAMILIES):
    p, cols = run_family(fam, VALID_START - pd.Timedelta(days=1), va_dates)
    rec_val_frame[cols] = p
    if (i + 1) % 11 == 0:
        print(f"  {i+1}/{len(FAMILIES)} families ({time.time()-t:.0f}s)", flush=True)
recursive_val = rec_val_frame.to_numpy().ravel()      # date-major, series-fastest
print(f"recursive half, holdout: {rmsle(y_va, np.expm1(recursive_val)):.5f}  "
      f"({time.time()-t:.0f}s)")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  11/33 families (17s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  22/33 families (33s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  33/33 families (51s)
recursive half, holdout: 0.39893  (51s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

In [8]:
print(f"direct    {rmsle(y_va, np.expm1(direct_val)):.5f}")
print(f"recursive {rmsle(y_va, np.expm1(recursive_val)):.5f}")
r_d = direct_val - np.log1p(y_va)
r_r = recursive_val - np.log1p(y_va)
corr = np.corrcoef(r_d, r_r)[0, 1]
print(f"residual correlation: {corr:.3f}   "
      f"(seed-to-seed within one model is 0.972-0.979 -- lower means genuinely different)")
print()
print("blend sweep (w = direct share):")
for w in (1.0, 0.9, 0.8, 0.75, 0.70, 0.65, 0.6, 0.5, 0.0):
    mark = "  <- shipping" if abs(w - BLEND_W) < 1e-9 else ""
    print(f"  w={w:.2f}  {rmsle(y_va, np.expm1(w*direct_val + (1-w)*recursive_val)):.5f}{mark}")
print()
print("per-day, to confirm the complementarity holds this run:")
rows = []
for day in (1, 2, 4, 8, 12, 16):
    sel = DAY_OF == day
    rows.append({"day": day,
                 "direct": rmsle(y_va[sel], np.expm1(direct_val[sel])),
                 "recursive": rmsle(y_va[sel], np.expm1(recursive_val[sel]))})
bd = pd.DataFrame(rows)
bd["recursive wins by"] = bd.direct - bd.recursive
print(bd.to_string(index=False, float_format=lambda v: f"{v: .5f}"))
print()
print("The recursive column must WORSEN with the forecast day -- that is compounding error, and")
print("its absence would mean the rollout is somehow seeing the future.")
assert bd.recursive.iloc[-1] > bd.recursive.iloc[0], "recursive half does not degrade with horizon"


direct    0.38899
recursive 0.39893
residual correlation: 0.917   (seed-to-seed within one model is 0.972-0.979 -- lower means genuinely different)

blend sweep (w = direct share):
  w=1.00  0.38899
  w=0.90  0.38697
  w=0.80  0.38562
  w=0.75  0.38519
  w=0.70  0.38494  <- shipping
  w=0.65  0.38485
  w=0.60  0.38493
  w=0.50  0.38560
  w=0.00  0.39893

per-day, to confirm the complementarity holds this run:
 day   direct  recursive  recursive wins by
   1  0.37930    0.38488           -0.00558
   2  0.36781    0.38063           -0.01281
   4  0.37223    0.38286           -0.01064
   8  0.39160    0.38629            0.00531
  12  0.40139    0.43411           -0.03272
  16  0.44508    0.41187            0.03321

The recursive column must WORSEN with the forecast day -- that is compounding error, and
its absence would mean the rollout is somehow seeing the future.


---
## Refit both halves on all history and write the submission

In [9]:
test_log_direct = {}
te_key = None
for min_lag in MIN_LAGS:
    t = time.time()
    d, cols = build_design(min_lag)
    eq_row = d.date.isin(eq_dates)
    full_tr = (d.date <= TRAIN_END) & d.target.notna() & ~eq_row
    te_m = d.date > TRAIN_END
    dfull = lgb.Dataset(d.loc[full_tr, cols], d.loc[full_tr,"target"],
                        categorical_feature=CATS, free_raw_data=False)
    logs = []
    for i in range(len(SEED_CFG)):
        mi = lgb.train(seed_params(i), dfull, num_boost_round=best_rounds[min_lag])
        logs.append(mi.predict(d.loc[te_m, cols]))
        del mi; gc.collect()
    test_log_direct[min_lag] = np.mean(logs, axis=0)
    if te_key is None:
        te_key = d.loc[te_m, ["date","store_nbr","family"]].copy()
    print(f"  direct lag>={min_lag:>2} refit ({time.time()-t:.0f}s)", flush=True)
    del d, dfull, logs; gc.collect()

te_day = ((te_key.date.to_numpy() - np.datetime64(TRAIN_END)) / np.timedelta64(1,"D")).astype(int)
assert te_day.min() == 1 and te_day.max() == 16
direct_test = np.empty(len(te_key))
for day, ml in ASSEMBLY.items():
    direct_test[te_day == day] = test_log_direct[ml][te_day == day]

  direct lag>= 1 refit (661s)
  direct lag>= 2 refit (486s)
  direct lag>= 4 refit (853s)
  direct lag>=16 refit (987s)


In [10]:
te_dates = pd.date_range(TRAIN_END + pd.Timedelta(days=1), test.date.max())
t = time.time()
rec_test_frame = pd.DataFrame(index=te_dates, columns=SERIES, dtype="float32")
for i, fam in enumerate(FAMILIES):
    p, cols = run_family(fam, TRAIN_END, te_dates)
    rec_test_frame[cols] = p
    if (i + 1) % 11 == 0:
        print(f"  {i+1}/{len(FAMILIES)} families ({time.time()-t:.0f}s)", flush=True)
recursive_test = rec_test_frame.to_numpy().ravel()
print(f"recursive refit + rollout done ({time.time()-t:.0f}s)")

# te_key is date-major with series varying fastest, which is exactly how rec_test_frame ravels.
# Assert it rather than trust it -- blending two misaligned vectors would produce a plausible
# looking number and a ruined submission.
exp_store = np.tile(SERIES.get_level_values(0).to_numpy(), len(te_dates))
exp_fam = np.tile(SERIES.get_level_values(1).to_numpy(), len(te_dates))
assert (te_key.store_nbr.to_numpy() == exp_store).all(), "row order mismatch (store)"
assert (te_key.family.astype(str).to_numpy() == exp_fam).all(), "row order mismatch (family)"
print("row alignment between the two halves verified")

print(f"mean log prediction -- direct {direct_test.mean():.3f} | "
      f"recursive {recursive_test.mean():.3f}")
assert abs(direct_test.mean() - recursive_test.mean()) < 1.0, \
    "the halves disagree on scale -- one of them did not train properly"

final_log = BLEND_W * direct_test + (1 - BLEND_W) * recursive_test
pred = np.clip(np.expm1(final_log), 0, None)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  11/33 families (17s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  22/33 families (34s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

  33/33 families (52s)
recursive refit + rollout done (52s)
row alignment between the two halves verified
mean log prediction -- direct 3.623 | recursive 3.624


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

In [11]:
# Product lines with no sales in the past year are forced to exactly zero. A pooled model
# cannot represent exact zero; under RMSLE these rows are 3.6% of the forecast, 0.01% of the
# units, and were worth -0.0153 on the leaderboard.
cutoff = TRAIN_END - pd.Timedelta(days=DORMANT_DAYS)
recent_total = (train[train.date > cutoff]
                .groupby(["store_nbr","family"], observed=True).sales.sum())
dead = {(s, str(f)) for (s, f), v in recent_total.items() if v == 0}
all_keys = {(s, str(f)) for s, f in zip(test.store_nbr, test.family.astype(str))}
dead |= all_keys - {(s, str(f)) for s, f in recent_total.index}
keys = list(zip(te_key.store_nbr.to_numpy(), te_key.family.astype(str).to_numpy()))
dormant = np.array([k in dead for k in keys])
leaked = pred[dormant]
print(f"dormant lines: {len(dead)} combinations, {dormant.sum():,} rows ({dormant.mean():.1%}), "
      f"carrying {leaked.sum():,.0f} units ({leaked.sum()/pred.sum():.3%} of volume)")
pred = pred.copy(); pred[dormant] = 0.0

out = te_key.copy(); out["sales"] = pred
out["family"] = out.family.astype(str)
key = test.assign(family=test.family.astype(str))[["id","date","store_nbr","family"]]
submission = key.merge(out, on=["date","store_nbr","family"], how="left")
assert len(submission) == len(test)
assert submission.sales.notna().all()
assert (submission.sales >= 0).all()
assert submission.id.equals(test.id)

volume = submission.sales.sum(); ratio = volume / VALIDATED_VOLUME
print(f"\ntotal predicted units : {volume:,.0f}")
print(f"last validated        : {VALIDATED_VOLUME:,.0f}  (ratio {ratio:.3f})")
assert 0.85 < ratio < 1.15, f"volume is {ratio:.2f}x the validated model's -- do not submit"

daily = submission.groupby("date").sales.sum()
print("\ndaily chain-wide forecast volume:")
print(daily.to_string(float_format=lambda v: f"{v:,.0f}"))
print(f"\nlargest day-over-day change: {daily.pct_change().abs().max():.1%} "
      "(the weekly cycle alone moves sales about this much)")
print("Check continuity before submitting -- the weekly rhythm should be the only visible pattern.")

submission[["id","sales"]].to_csv(OUT / "submission.csv", index=False)
print(f"\nwritten -> {(OUT / 'submission.csv').resolve()}")
print(f"blend weight: {BLEND_W} direct / {1-BLEND_W:.2f} recursive")

dormant lines: 65 combinations, 1,040 rows (3.6%), carrying 428 units (0.003% of volume)

total predicted units : 12,623,173
last validated        : 12,635,225  (ratio 0.999)

daily chain-wide forecast volume:
date
2017-08-16     811,489
2017-08-17     640,038
2017-08-18     763,716
2017-08-19     890,751
2017-08-20   1,022,504
2017-08-21     798,281
2017-08-22     725,541
2017-08-23     760,527
2017-08-24     638,678
2017-08-25     752,997
2017-08-26     906,073
2017-08-27     999,169
2017-08-28     758,813
2017-08-29     696,847
2017-08-30     766,866
2017-08-31     690,884

largest day-over-day change: 24.1% (the weekly cycle alone moves sales about this much)
Check continuity before submitting -- the weekly rhythm should be the only visible pattern.

written -> /kaggle/working/submission.csv
blend weight: 0.7 direct / 0.30 recursive


---
## Reading the result

| Reference | Leaderboard |
|---|---|
| + per-horizon models | 0.41113 |
| + dormant-series zero | 0.39586 |
| **+ recursive blend** | **0.39079** ← the number this must beat |
| + mixed L2+Huber (rejected) | 0.39320 |
| this notebook (`rmean_3` in the near buckets) | ? |

**What was measured, on the rows the change serves** (days 1–4, paired, 5 production seeds):
−0.00244, sd 0.00100, **5/5 seeds, 5.5 σ**, and the 5-seed ensemble keeps −0.00236 of it.
Diluted over the whole window (4 of 16 days) and the blend (the direct half carries 70%), the
expected local effect is roughly **−0.0005**.

**How it might transfer.** This is *new information* — no near-bucket feature previously
carried days −1..−3 — and that category has transferred at or above its measured value every
time here (per-horizon 4.4×, promo rework sign-reversed in our favour, dormant zero 5.6×). A
reasonable range is therefore **−0.0005 to −0.002** on the leaderboard. For calibration,
environment noise between Kaggle's LightGBM build and the local one has been ~0.0004, so the
low end of that range is only just distinguishable from nothing.

**Why this is not another Huber.** The mixed-objective submission (0.39320, rejected) failed
from the 1.3 σ evidence band, from being fit optimisation, and from a gain that seed-ensembling
had already absorbed. This change sits in the other band on all three: 5.5 σ, new information,
and the ensemble keeps the gain. Those are exactly the three checks the Huber failure
established — they are stated here so the result can be read against them.

**What is deliberately NOT changed**, so the A/B stays clean: the far bucket (`lag ≥ 16` keeps
60 features — `rmean_3` there was measured to cancel the near-bucket gain), the recursive half,
the 70/30 weight, the five seeds, the 365-day dormancy rule, and all three submission guards.

If this lands at or above 0.39079, the conclusion is that a real near-horizon gain was too
diluted to survive the far days that dominate the score — and the 0.39079 submission stands.
